<a href="https://colab.research.google.com/github/andiunsia/Latihan_Data_Science/blob/main/Pertemuan10_Andi_240401010009.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Import Library

In [1]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')


# Upload dan Baca Dataset

In [12]:
import urllib.request

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv",
    "telco_churn.csv"
)
print("Dataset berhasil didownload: telco_churn.csv")

df = pd.read_csv('/content/telco_churn.csv')

print("Ukuran dataset:", df.shape)
df.head()
print(df.shape)

Dataset berhasil didownload: telco_churn.csv
Ukuran dataset: (7043, 21)
(7043, 21)


# Explorasi Data

In [4]:
print(df.info())
print(df.isnull().sum())

# Proporsi kelas Churn
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


# Preprocessing

In [13]:
# Hapus costumer ID
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

# Konversi target menjadi numerik
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df.dropna(subset=['Churn'], inplace=True)

# Menangani kolom TotalCharges
if 'TotalCharges' in df.columns:
    df['TotalCharges'] = pd.to_numeric(
        df['TotalCharges'],
        errors='coerce'
    )

    df['TotalCharges'].fillna(
        df['TotalCharges'].median(),
        inplace=True
    )

# One-Hot Encoding
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

# Pisahkan fitur dan target
X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train:", X_tr.shape)
print("Test :", X_te.shape)

Train: (5634, 30)
Test : (1409, 30)


# Latih Model

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_tr, y_tr)

RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

# Evaluasi Model

In [15]:
# Prediksi
y_pred = rf.predict(X_te)
y_proba = rf.predict_proba(X_te)[:,1]

# Classification Report
print(classification_report(y_te, y_pred))

# Confusion Matrix
print(confusion_matrix(y_te, y_pred))

# ROC-AUC
roc = roc_auc_score(y_te, y_proba)
print("ROC-AUC:", round(roc, 4))

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.63      0.50      0.56       374

    accuracy                           0.79      1409
   macro avg       0.73      0.70      0.71      1409
weighted avg       0.78      0.79      0.78      1409

[[925 110]
 [187 187]]
ROC-AUC: 0.8246


# Prediksi Probabilitas Churn

In [18]:
# Membuat probabilitas churn untuk setiap pelanggan pada data uji
hasil = pd.DataFrame({
    "Aktual": y_te.values,
    "Prediksi": y_pred,
    "Probabilitas_Churn": y_proba
})

hasil.head(10)

# Melihat pelanggan dengan risiko churn tertinggi
hasil.sort_values(
    by="Probabilitas_Churn",
    ascending=False
).head(10)

top10 = hasil.sort_values(
    by="Probabilitas_Churn",
    ascending=False
).head(10)

print("=== 10 Pelanggan Dengan Risiko Churn Tertinggi ===")
print(top10)

print("\nJumlah pelanggan data uji :", len(hasil))
print("Probabilitas churn tertinggi :", round(hasil["Probabilitas_Churn"].max(),4))
print("Probabilitas churn terendah :", round(hasil["Probabilitas_Churn"].min(),4))
print("Probabilitas churn rata-rata :", round(hasil["Probabilitas_Churn"].mean(),4))


=== 10 Pelanggan Dengan Risiko Churn Tertinggi ===
      Aktual  Prediksi  Probabilitas_Churn
1289       1         1            1.000000
171        1         1            0.993333
341        1         1            0.990000
618        1         1            0.990000
1252       1         1            0.986667
629        0         1            0.986667
889        0         1            0.970000
1178       1         1            0.963333
1109       1         1            0.960000
788        1         1            0.950000

Jumlah pelanggan data uji : 1409
Probabilitas churn tertinggi : 1.0
Probabilitas churn terendah : 0.0
Probabilitas churn rata-rata : 0.2681


In [19]:

# Mengetahui faktor yang paling mempengaruhi churn
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance.sort_values(
    by="Importance",
    ascending=False
).head(10)

,Feature,Importance
3,TotalCharges,0.177844
1,tenure,0.164403
2,MonthlyCharges,0.151054
25,Contract_Two year,0.059944
10,InternetService_Fiber optic,0.042323
28,PaymentMethod_Electronic check,0.036455
24,Contract_One year,0.029412
13,OnlineSecurity_Yes,0.028447
4,gender_Male,0.025604
26,PaperlessBilling_Yes,0.024087


# Kesimpulan

Model Random Forest berhasil digunakan untuk memprediksi kemungkinan customer churn pada dataset Telco Customer Churn. Penggunaan parameter class_weight='balanced' membantu menangani ketidakseimbangan kelas karena pelanggan churn hanya sekitar 26,5% dari total data. Hasil evaluasi menggunakan precision, recall, F1-score, dan ROC-AUC menunjukkan bahwa model mampu membedakan pelanggan churn dan non-churn dengan cukup baik. Probabilitas churn yang dihasilkan dapat dimanfaatkan tim bisnis untuk mengidentifikasi pelanggan berisiko tinggi dan melakukan strategi retensi secara lebih efektif.